Antes de rodar, crie um .env na raiz do projeto com:
DB_HOST=localhost
DB_NAME=exercicios_colaborativos
DB_USER=postgres
DB_PASSWORD=sua_senha

In [1]:
!pip install pandas sqlalchemy psycopg2-binary panel python-dotenv jupyter_bokeh

import os
from dotenv import load_dotenv
import pandas as pd
import psycopg2 as pg
import sqlalchemy
from sqlalchemy import create_engine
import panel as pn
import matplotlib.pyplot as plt

# Inicializa as extensões do Panel na sessão do Jupyter
pn.extension('tabulator', 'matplotlib', notifications=True)


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: C:\Users\João Pedro\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable


BokehModel(combine_events=True, render_bundle={'docs_json': {'cd79fce0-e2d5-47a6-a0db-a8eca3b14797': {'version…

In [2]:
load_dotenv()

# Lê as variáveis de ambiente
DB_HOST = os.getenv('DB_HOST')
DB_NAME = os.getenv('DB_NAME')
DB_USER = os.getenv('DB_USER')
DB_PASS = os.getenv('DB_PASSWORD')

# Cria a string de conexão para o PostgreSQL
cnx = f'postgresql://{DB_USER}:{DB_PASS}@{DB_HOST}/{DB_NAME}'
engine = create_engine(cnx)

print("Conexão configurada com sucesso!")

Conexão configurada com sucesso!


In [3]:
# IDs de Controle e Autoria
id_denuncia_alvo = pn.widgets.IntInput(name="ID da Denúncia (para Atualizar/Excluir)", value=0)
id_autor = pn.widgets.IntInput(name="ID do Autor da Denúncia *", value=0)

# Dados da Denúncia
motivo = pn.widgets.TextAreaInput(name="Motivo da Denúncia *", value='', placeholder='Descreva o motivo...')
status_analise = pn.widgets.Select(name="Status da Análise *", options=['Pendente', 'Em Análise', 'Aprovada (Removido)', 'Rejeitada'])

# Vínculos Obrigatórios (Regra de Negócio: Pergunta OU Resposta)
id_pergunta = pn.widgets.IntInput(name="ID da Pergunta Denunciada (0 se não houver)", value=0)
id_resposta = pn.widgets.IntInput(name="ID da Resposta Denunciada (0 se não houver)", value=0)

# Botões de Ação
buttonConsultar = pn.widgets.Button(name='Consultar Denúncias', button_type='default')
buttonInserir = pn.widgets.Button(name='Registrar Denúncia', button_type='primary')
buttonAtualizar = pn.widgets.Button(name='Atualizar Status', button_type='warning')
buttonExcluir = pn.widgets.Button(name='Excluir Registro', button_type='danger')

In [4]:
def on_consultar():
    query = "SELECT * FROM Denuncia ORDER BY data_denuncia DESC;"
    try:
        df = pd.read_sql_query(query, engine)
        return pn.widgets.Tabulator(df, pagination='remote', page_size=10, theme='bootstrap')
    except Exception as e:
        pn.state.notifications.error(f"Erro ao consultar: {e}")
        return "Erro ao carregar dados do banco."

def on_inserir():
    if id_pergunta.value == 0 and id_resposta.value == 0:
        pn.state.notifications.error("A denúncia deve estar vinculada a uma pergunta ou resposta!")
        return on_consultar()
    
    p_id = "NULL" if id_pergunta.value == 0 else id_pergunta.value
    r_id = "NULL" if id_resposta.value == 0 else id_resposta.value

    query = f"""
        INSERT INTO Denuncia (id_autor, motivo, data_denuncia, status_analise, id_pergunta, id_resposta)
        VALUES ({id_autor.value}, '{motivo.value}', CURRENT_TIMESTAMP, '{status_analise.value}', {p_id}, {r_id});
    """
    try:
        with engine.begin() as conn:
            conn.execute(sqlalchemy.text(query))
        pn.state.notifications.success("Denúncia registrada com sucesso!")
    except Exception as e:
        pn.state.notifications.error(f"Erro ao inserir: {e}")
    return on_consultar()

def on_atualizar():
    if id_denuncia_alvo.value == 0:
        pn.state.notifications.error("Informe um ID de denúncia válido.")
        return on_consultar()
        
    query = f"UPDATE Denuncia SET status_analise = '{status_analise.value}' WHERE id_denuncia = {id_denuncia_alvo.value};"
    try:
        with engine.begin() as conn:
            conn.execute(sqlalchemy.text(query))
        pn.state.notifications.success("Status atualizado!")
    except Exception as e:
        pn.state.notifications.error(f"Erro ao atualizar: {e}")
    return on_consultar()

def on_excluir():
    if id_denuncia_alvo.value == 0:
        pn.state.notifications.error("Informe um ID de denúncia válido.")
        return on_consultar()
        
    query = f"DELETE FROM Denuncia WHERE id_denuncia = {id_denuncia_alvo.value};"
    try:
        with engine.begin() as conn:
            conn.execute(sqlalchemy.text(query))
        pn.state.notifications.success("Denúncia removida.")
    except Exception as e:
        pn.state.notifications.error(f"Erro ao excluir: {e}")
    return on_consultar()

# Gerencia qual ação tomar baseado no clique dos botões
def table_creator(cons, ins, atu, exc):
    return on_consultar()

# Vincula a tabela interativa aos eventos dos botões
interactive_table = pn.bind(table_creator, buttonConsultar, buttonInserir, buttonAtualizar, buttonExcluir)

In [5]:
def criar_graficos_pandas(event=None):
    query_status = "SELECT status_analise, COUNT(*) as qtd FROM Denuncia GROUP BY status_analise"
    
    try:
        df_status = pd.read_sql(query_status, engine)
    except:
        df_status = pd.DataFrame(columns=['status_analise', 'qtd'])

    fig1, ax1 = plt.subplots(figsize=(5, 3.5))

    if not df_status.empty:
        df_status.set_index('status_analise').plot(
            kind='bar', y='qtd', ax=ax1, color='#d9534f', rot=25, legend=False, title='Volume por Status'
        )
        ax1.set_ylabel('Quantidade')
    else:
        ax1.text(0.5, 0.5, "Sem dados cadastrados", ha='center')

    plt.close(fig1)
    return pn.Column(pn.pane.Matplotlib(fig1, tight=True), name="Métricas")

In [6]:
# Montagem da aba de gerenciamento (CRUD)
layout_crud = pn.Row(
    pn.Column(
        '### Painel de Moderação de Conteúdo (RF09)',
        id_denuncia_alvo, id_autor, motivo, status_analise,
        '**Vínculo (Preencha pelo menos um)**',
        id_pergunta, id_resposta,
        pn.Row(buttonConsultar, buttonInserir),
        pn.Row(buttonAtualizar, buttonExcluir),
        width=320
    ),
    pn.Column(interactive_table),
    name="Moderar Conteúdo"
)

# Vincula a atualização do dashboard ao botão de inserção
layout_dashboard = pn.bind(criar_graficos_pandas, buttonInserir)

# Junção das abas
aba_principal = pn.Tabs(
    layout_crud,
    ("Dashboard Moderação", layout_dashboard)
)

# Renderiza o painel diretamente no Notebook
aba_principal

BokehModel(combine_events=True, render_bundle={'docs_json': {'9016eae8-d08b-4ff4-b965-1debf59067b9': {'version…